In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7703244728212504, 'n_it': 0.3973975080734152}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[17.120116895982047, 17.473397079933797, 18.139925797936492, 13.515553117078486, 15.891485554688714, 13.441203091423757, 16.750110816132135, 13.396055436655745, 13.925701560650262, 16.44872900154707, 15.308180733642125, 13.479826933635518, 13.883589890114086, 13.712994435769577, 13.597272249299284, 16.32829833348382, 18.021585035503517, 13.608336811372277, 16.650116359532504, 14.212753488655878, 15.620811198422786, 13.560782241380117, 14.16951806829543, 14.102467311904775, 13.422213442247475, 13.815735306359, 15.640544812259947, 17.94055475986616, 14.918438678579388, 18.27724150240276, 17.459376433831157, 13.349833994494388, 13.581267928126879, 13.46883534819471, 13.447977139100582, 14.405314051325554, 13.519688938198334, 15.461992117300154, 13.588391934952085, 14.391935087745365, 13.71644412627245, 14.346264203623923, 14.828557802782143, 15.111466523248444, 16.5105805805245, 14.323540373771108, 16.827054716045833, 18.23897014741191, 18.01134773445495, 16.041789997574003, 12.5714526297

In [5]:
np.average(y_max_arr)

np.float64(14.985432827877172)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)